# 04 — Sinh câu hỏi từ PDF người dùng tải lên, với Writer là mô hình đã fine-tune

Pipeline gốc chạy **nguyên trạng** (phương án nhiễu → giải độc lập → kiểm chứng → chấm → đóng gói),
chỉ thay tác nhân **Writer**: thay vì model thị giác đọc ảnh trang PDF, câu hỏi do mô hình đã fine-tune soạn.

Qwen3-8B/14B chỉ đọc chữ, và ba mô hình không cùng vừa trong 80GB, nên chạy **ba giai đoạn**, mỗi giai đoạn một tiến trình:

| Giai đoạn | Mô hình trên GPU | Việc |
|---|---|---|
| 1. `prepare` | model thị giác | chép từng trang PDF thành chữ (công thức sang LaTeX) + trích dàn ý (chủ đề, chuẩn đầu ra) |
| 2. `draft` | mô hình fine-tune | soạn sẵn kho câu nháp theo mức độ, lấy trích đoạn tài liệu làm ngữ cảnh; lọc trước bằng bộ luật của pipeline |
| 3. `generate` | model thị giác + solver | pipeline duyệt từng câu nháp: 3 phương án nhiễu gắn lỗi, giải lại độc lập, kiểm chứng, chấm bám nguồn, đóng gói |

Cần: **A100 80GB**, `HF_TOKEN` trong Colab Secrets, và một adapter đã train (notebook 01 hoặc 02).

Ba điểm khác với Writer gốc, cần biết khi đọc kết quả:

- **Trích dẫn nguồn** do máy dò lại trong bản chép của tài liệu (mô hình fine-tune không được huấn luyện để trích dẫn). Critic vẫn chấm lại độ bám nguồn trên ảnh trang gốc.
- **Không có biểu thức kiểm chứng**, nên SymPy không tự tính lại được; nhãn kiểm chứng đến từ hội đồng giải độc lập.
- **Ràng buộc độ nặng phép tính** của Writer gốc không áp được (mô hình chỉ nhận mức độ), nên câu dễ hơn mong đợi sẽ bị bộ luật "đề quá ngắn/quá dễ" của pipeline loại — xem cột lý do loại ở cuối.

In [ ]:
#@title 1. Kiểm tra GPU
import subprocess
info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                       '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
print(info)
name, mem = [x.strip() for x in info.split(',')]
assert 'A100' in name and int(mem) >= 79000, (
    f'Ngân sách bộ nhớ trong notebook tính cho A100 80GB, runtime hiện tại: {name} {mem} MiB')

In [ ]:
#@title 2. Cấu hình
ADAPTER_RUN = 'expB_qwen3_14b_seed42'  #@param {type:'string'}
#@markdown ↑ thư mục thí nghiệm trong Drive (notebook 01 hoặc 02 đã tạo)
BASE_MODEL = 'Qwen/Qwen3-14B'  #@param ['Qwen/Qwen3-14B', 'Qwen/Qwen3-8B']
PRESET = 'quality'  #@param ['quality', 'fast']
N_QUESTIONS = 10  #@param {type:'integer'}
BLOOM = 'mixed'  #@param ['mixed', 'Nhận biết', 'Thông hiểu', 'Vận dụng', 'Vận dụng cao']
OVERSAMPLE = 4  #@param {type:'integer'}
DISTRACTORS = 'pipeline'  #@param ['pipeline', 'model']
#@markdown ↑ `pipeline`: tác nhân nhiễu của pipeline soạn lại 3 phương án gắn lỗi (khuyên dùng).
#@markdown `model`: giữ 4 phương án của mô hình fine-tune, nhưng không có mô tả lỗi nên Critic không kiểm được tính nhất quán lỗi↔giá trị.
PDF_PATH = ''  #@param {type:'string'}
#@markdown ↑ để trống thì ô 5 cho tải file lên
INCLUDE_EXPLANATION = True  #@param {type:'boolean'}
RUN_BASELINE = False  #@param {type:'boolean'}
#@markdown ↑ chạy thêm pipeline gốc (Writer là model thị giác) trên cùng PDF để so sánh
REPO = 'https://github.com/trantrien1/AQG.git'  #@param {type:'string'}
BRANCH = 'lora-finetune'  #@param {type:'string'}
DRIVE_ROOT = '/content/drive/MyDrive/AQG_ft'  #@param {type:'string'}

import json, time
PRESETS = {
    # Model thị giác 32B FP8 cho chất lượng đọc trang tốt nhất (như notebook colab của pipeline).
    'quality': dict(vision='Qwen/Qwen3-VL-32B-Instruct-FP8', vision_util=0.60, vision_len=32768,
                    solver='microsoft/phi-4', solver_util=0.22, solver_len=4096,
                    solver_extra=['--quantization', 'fp8', '--enforce-eager'],
                    dpi=96, max_pages=24, parallel=3),
    # Nhanh hơn, nhẹ hơn: model thị giác 8B bf16.
    'fast': dict(vision='Qwen/Qwen3-VL-8B-Instruct', vision_util=0.42, vision_len=32768,
                 solver='microsoft/phi-4', solver_util=0.40, solver_len=4096,
                 solver_extra=['--enforce-eager'],
                 dpi=110, max_pages=30, parallel=4),
}
P = PRESETS[PRESET]
VISION_PORT, SOLVER_PORT, WRITER_PORT = 8000, 8001, 8002
RUN_NAME = 'pdf-' + time.strftime('%Y%m%d-%H%M')
print(PRESET, json.dumps(P, indent=1))

In [ ]:
#@title 3. Drive, token Hugging Face, hàm chạy lệnh
import os, sys, json, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')

try:  # Colab: 🔑 Secrets -> HF_TOKEN
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('Chưa đọc được HF_TOKEN từ Colab Secrets:', exc)

REPO_DIR = '/content/AQG'
API_DIR = f'{REPO_DIR}/API'
NB_DIR = f'{API_DIR}/notebooks'
ADAPTER = f'{DRIVE_ROOT}/{ADAPTER_RUN}/adapter'
WORK = f'/content/{RUN_NAME}'
SAVE = f'{DRIVE_ROOT}/{RUN_NAME}'
os.makedirs(WORK, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)

def sh(cmd, cwd=None, log=None):
    """Chạy lệnh, in đầu ra ngay khi có; lỗi thì dừng notebook."""
    print('$', cmd, flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (NB_DIR if os.path.isdir(NB_DIR) else None),
                         env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace')
    for chunk in iter(lambda: p.stdout.read(512), ''):
        sys.stdout.write(chunk)
        if fh:
            fh.write(chunk)
    if fh:
        fh.close()
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}): {cmd}')

ENV = dict(os.environ, PYTHONPATH=f'{NB_DIR}:{API_DIR}', TOKENIZERS_PARALLELISM='false',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', PYTHONIOENCODING='utf-8')

In [ ]:
#@title 4. Clone repo + cài thư viện (~5 phút)
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
sh(f'git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR} && git -C {REPO_DIR} log --oneline -1', cwd='/content')
sh('pip -q install -U vllm && pip -q install -U PyMuPDF networkx sympy openpyxl python-docx pytest', cwd='/content')
sh('python -m pytest -q ../tests/test_mcqft.py ../tests/test_mcqft_pipeline.py')
assert os.path.isdir(ADAPTER), f'Không thấy adapter: {ADAPTER} — chạy notebook 01/02 trước'
print('adapter:', ADAPTER, os.listdir(ADAPTER))

In [ ]:
#@title 5. Chọn PDF (tải lên hoặc lấy từ Drive)
if PDF_PATH.strip():
    PDF = PDF_PATH.strip()
else:
    from google.colab import files
    up = files.upload()
    name = list(up)[0]
    PDF = f'{WORK}/{name}'
    with open(PDF, 'wb') as fh:
        fh.write(up[name])
assert os.path.exists(PDF), PDF
import fitz
doc = fitz.open(PDF)
print(f'{PDF}: {len(doc)} trang, {os.path.getsize(PDF) / 1e6:.1f} MB')
shutil.copy(PDF, SAVE)

In [ ]:
#@title 6. Ghim phiên bản trọng số + biến môi trường cho pipeline
import secrets
from huggingface_hub import HfApi

api = HfApi()
REVISIONS = {m: api.model_info(m).sha for m in (P['vision'], P['solver'], BASE_MODEL)}
print(json.dumps(REVISIONS, indent=1))
VLLM_KEY = os.environ.get('VLLM_KEY') or secrets.token_urlsafe(24)
VISION_URL = f'http://127.0.0.1:{VISION_PORT}/v1'
SOLVER_URL = f'http://127.0.0.1:{SOLVER_PORT}/v1'
PIPELINE_ENV = {
    'VLLM_KEY': VLLM_KEY,
    'AQG_LLM_PROVIDER': 'openai_compatible',
    'OPENAI_COMPATIBLE_BASE_URL': VISION_URL,
    'OPENAI_API_KEY': VLLM_KEY,
    'OPENAI_GENERATOR_MODEL': P['vision'],
    'OPENAI_JUDGE_MODEL': P['vision'],
    # Hội đồng giải độc lập: phi-4 (khác họ) + chính model thị giác; luật 'all'.
    'AQG_INDEPENDENT_SOLVERS': f"{P['solver']}@{SOLVER_URL},{P['vision']}@{VISION_URL}",
    'AQG_INDEPENDENT_API_KEY': VLLM_KEY,
    'AQG_INDEPENDENT_CONSENSUS': 'all',
    'AQG_MODEL_REVISIONS': json.dumps(REVISIONS),
    'AQG_PDF_ATTACH_MODE': 'image',
    'AQG_PDF_IMAGE_DPI': str(P['dpi']),
    'AQG_PDF_IMAGE_MAX_PAGES': str(P['max_pages']),
    'AQG_ATTACHMENTS_FIRST': '1',
    'AQG_DIRECT_PDF_PARALLEL_SLOTS': str(P['parallel']),
    'AQG_LLM_TIMEOUT_SECONDS': '900',
    'AQG_LLM_RETRIES': '2',
}
os.environ.update(PIPELINE_ENV)
ENV.update(PIPELINE_ENV)
json.dump({k: v for k, v in PIPELINE_ENV.items() if 'KEY' not in k},
          open(f'{SAVE}/pipeline_env.json', 'w'), indent=1)

In [ ]:
#@title 7. Hàm bật/tắt máy chủ vLLM
import time, urllib.request
LOG_DIR = f'{WORK}/vllm_logs'
os.makedirs(LOG_DIR, exist_ok=True)
SERVERS = globals().get('SERVERS', {})

def start_vllm(name, model, port, util, max_len, extra=(), vision=False, lora=None):
    if name in SERVERS and SERVERS[name].poll() is None:
        print(f'{name} đang chạy (pid {SERVERS[name].pid})'); return
    cmd = ['vllm', 'serve', model, '--host', '127.0.0.1', '--port', str(port),
           '--api-key', VLLM_KEY, '--gpu-memory-utilization', str(util),
           '--max-model-len', str(max_len), '--max-num-seqs', '8',
           '--enable-prefix-caching'] + list(extra)
    if model in REVISIONS:
        cmd += ['--revision', REVISIONS[model], '--tokenizer-revision', REVISIONS[model]]
    if vision:
        cmd += ['--limit-mm-per-prompt', json.dumps({'image': P['max_pages'] + 2, 'video': 0})]
    if lora:
        cmd += ['--enable-lora', '--max-lora-rank', '32', '--lora-modules', f'aqg-lora={lora}']
    log = open(f'{LOG_DIR}/{name}.log', 'w')
    SERVERS[name] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                     start_new_session=True, env=ENV)
    print(f'bật {name}: {model} (pid {SERVERS[name].pid}), log {LOG_DIR}/{name}.log')

def wait_ready(name, port, timeout=2400):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if SERVERS[name].poll() is not None:
            print(open(f'{LOG_DIR}/{name}.log', encoding='utf-8', errors='replace').read()[-4000:])
            raise RuntimeError(f'{name} đã thoát (mã {SERVERS[name].returncode}) — xem log ở trên')
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=5) as r:
                if r.status == 200:
                    print(f'{name} sẵn sàng sau {time.time() - t0:.0f}s'); return
        except Exception:
            pass
        time.sleep(15)
    raise TimeoutError(f'{name} chưa sẵn sàng sau {timeout}s')

def stop_all():
    for name, proc in list(SERVERS.items()):
        if proc.poll() is None:
            os.killpg(proc.pid, 15)
            try:
                proc.wait(timeout=180)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, 9)
            print('dừng', name)
    SERVERS.clear()
    time.sleep(5)
    print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv'],
                         capture_output=True, text=True).stdout.strip())

## Giai đoạn 1 — model thị giác chép tài liệu thành chữ

Mỗi trang một lời gọi: giữ nguyên chữ, công thức sang LaTeX, bảng thành dòng, hình ghi `[Hình: ...]`.
Thêm một lời gọi trích dàn ý (chủ đề + chuẩn đầu ra gợi ý) — chính bước `/prepare` của pipeline.

In [ ]:
#@title 8. Bật model thị giác rồi chép tài liệu (~5–15 phút)
start_vllm('vision', P['vision'], VISION_PORT, P['vision_util'], P['vision_len'], vision=True)
wait_ready('vision', VISION_PORT)
sh(f'python -m mcqft.pipeline_ft prepare --pdf "{PDF}" --out "{WORK}" '
   f'--model {P["vision"]} --dpi 144 --max-pages {P["max_pages"]} --workers {P["parallel"]}',
   log=f'{SAVE}/prepare.log')
shutil.copy(f'{WORK}/prep.json', SAVE)

In [ ]:
#@title 9. Xem bản chép trang đầu + dàn ý
prep = json.load(open(f'{WORK}/prep.json', encoding='utf-8'))
print('DÀN Ý:', json.dumps(prep['outline'], ensure_ascii=False, indent=1))
print('\n--- TRANG 1 ---\n')
print(prep['pages'][0][:2000])

In [ ]:
#@title 10. Tắt máy chủ để nhường GPU cho mô hình fine-tune
stop_all()

## Giai đoạn 2 — mô hình fine-tune soạn kho câu nháp

Mỗi câu nháp được sinh từ một trích đoạn tài liệu + một mức độ, đúng định dạng đã học ở tác vụ `gen_ctx`.
Kho được soạn dư (`OVERSAMPLE` lần) vì phần lớn câu sẽ bị lọc: sai khuôn, chép lại bài trong tài liệu,
trùng nhau, hoặc không qua bộ luật soạn đề của pipeline (đề quá ngắn, phương án trùng, lời giải thiếu bước…).

In [ ]:
#@title 11. Soạn kho câu nháp (~3–8 phút)
sh(f'python -m mcqft.pipeline_ft draft --prep "{WORK}" --model {BASE_MODEL} '
   f'--revision {REVISIONS[BASE_MODEL]} --adapter "{ADAPTER}" --n {N_QUESTIONS} '
   f'--oversample {OVERSAMPLE} --bloom "{BLOOM}" --distractors {DISTRACTORS} --gpu-util 0.85',
   log=f'{SAVE}/draft.log')
for f in ('pool.jsonl', 'draft_summary.json'):
    shutil.copy(f'{WORK}/{f}', SAVE)

In [ ]:
#@title 12. Xem một câu nháp và lý do các câu khác bị lọc
from IPython.display import Markdown, display
summary = json.load(open(f'{WORK}/draft_summary.json', encoding='utf-8'))
display(Markdown('**Bị lọc:**\n\n' + '\n'.join(
    f'- {k}: {v}' for k, v in sorted(summary['dropped'].items(), key=lambda kv: -kv[1]))))
pool = [json.loads(l) for l in open(f'{WORK}/pool.jsonl', encoding='utf-8')]
if pool:
    display(Markdown(f"**Câu nháp mẫu** (mức {pool[0]['level']}, chủ đề {pool[0]['topic']!r}, "
                     f"trang {pool[0]['pages']}):\n\n" + pool[0]['text']))

## Giai đoạn 3 — pipeline duyệt từng câu nháp

Bật **solver trước** (phi-4 FP8 nạp trọng số bf16 rồi mới lượng tử hoá nên cần GPU còn trống), rồi model thị giác.
Mỗi câu đi qua: phương án nhiễu gắn lỗi (đọc trang PDF) → hội đồng giải độc lập → kiểm chứng + luật soạn đề →
chấm bám nguồn và rubric → đóng gói (trộn vị trí đáp án, ước lượng độ khó, phân luồng người duyệt).

In [ ]:
#@title 13. Bật solver + model thị giác
start_vllm('solver', P['solver'], SOLVER_PORT, P['solver_util'], P['solver_len'], P['solver_extra'])
wait_ready('solver', SOLVER_PORT)
start_vllm('vision', P['vision'], VISION_PORT, P['vision_util'], P['vision_len'], vision=True)
wait_ready('vision', VISION_PORT)
print(subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
#@title 14. Sinh câu hỏi (~1–3 phút mỗi câu)
opts = '' if INCLUDE_EXPLANATION else ' --no-explanation'
sh(f'python -m mcqft.pipeline_ft generate --prep "{WORK}" --pdf "{PDF}" --n {N_QUESTIONS} '
   f'--bloom "{BLOOM}" --distractors {DISTRACTORS} --out "{WORK}/questions"{opts}',
   log=f'{SAVE}/generate.log')
shutil.copytree(f'{WORK}/questions', f'{SAVE}/questions', dirs_exist_ok=True)

In [ ]:
#@title 15. Kết quả
from IPython.display import Markdown, display
out = json.load(open(f'{WORK}/questions/questions.json', encoding='utf-8'))
summary = json.load(open(f'{WORK}/questions/summary.json', encoding='utf-8'))
display(Markdown(
    f"**{summary['accepted_questions']}/{summary['requested_questions']} câu** trong "
    f"{summary['seconds']:.0f}s · nhãn kiểm chứng {summary['status_counts']} · "
    f"trạng thái duyệt {summary['review_status']} · mức độ {summary['levels']}\n\n"
    f"Writer: {summary['writer']['stats']}\n\n"
    f"Lý do loại: {summary['reject_codes']}"))

for q in out['questions']:
    body = [f"### {q['cognitive_level']} · {(q.get('verification') or {}).get('status')} "
            f"· {q['review_status']}", '', q['stem'], '']
    for o in q['options']:
        mark = '**(đáp án)**' if o['key'] == q['answer_key'] else ''
        body.append(f"- **{o['key']}.** {o['text']} {mark}")
    body += ['', f"**Giải thích:** {q.get('explanation_correct', '')}", '',
             f"**Trích dẫn nguồn:** {(q.get('source') or {}).get('quote', '')}"]
    display(Markdown('\n'.join(body)))

In [ ]:
#@title 16. Vì sao các câu khác bị loại
rejected = [json.loads(l) for l in open(f'{WORK}/questions/rejected.jsonl', encoding='utf-8')]
from collections import Counter
print(Counter(r.get('reject_reason_code') for r in rejected))
for r in rejected[:5]:
    print('---', r.get('reject_reason_stage'), r.get('reject_reason_code'))
    print((r.get('stem') or '')[:200])
    print((r.get('reject_reason') or '')[:300])

In [ ]:
#@title 17. (Tuỳ chọn) Đối chứng: pipeline gốc với Writer là model thị giác
if RUN_BASELINE:
    sh(f'python run.py --pdf "{PDF}" -n {N_QUESTIONS} --bloom-level "{BLOOM}" '
       f'--output "{WORK}/baseline.json"', cwd=API_DIR, log=f'{SAVE}/baseline.log')
    shutil.copy(f'{WORK}/baseline.json', SAVE)
    base = json.load(open(f'{WORK}/baseline.json', encoding='utf-8'))
    from collections import Counter
    print('Writer gốc :', base['metadata']['accepted_questions'], 'câu |',
          Counter((q.get('verification') or {}).get('status') for q in base['questions']))
    print('Writer FT  :', summary['accepted_questions'], 'câu |', summary['status_counts'])

In [ ]:
#@title 18. Lưu Drive + tắt máy chủ
shutil.copytree(LOG_DIR, f'{SAVE}/vllm_logs', dirs_exist_ok=True)
print('Đã lưu:', SAVE, os.listdir(SAVE))
stop_all()

## (Tuỳ chọn) Chế độ trực tuyến — cho web app

Bật thêm một máy chủ phục vụ base + adapter rồi bỏ giai đoạn 2: Writer gọi máy chủ đó cho **từng** slot,
nên tôn trọng được danh sách câu cần né và phản hồi người dùng như Writer gốc.

Chỉ đủ bộ nhớ khi Writer là **8B** và preset là `fast`: writer 0.26 + model thị giác 0.40 + solver 0.26.
Với 14B thì dùng ba giai đoạn ở trên.

In [ ]:
#@title 19. (Tuỳ chọn) Máy chủ writer + sinh trực tuyến
ONLINE = False  #@param {type:'boolean'}
if ONLINE:
    assert BASE_MODEL.endswith('8B') and PRESET == 'fast', 'Chế độ này dành cho writer 8B + preset fast'
    stop_all()
    start_vllm('writer', BASE_MODEL, WRITER_PORT, 0.26, 8192, lora=ADAPTER)
    wait_ready('writer', WRITER_PORT)
    start_vllm('solver', P['solver'], SOLVER_PORT, 0.26, P['solver_len'], P['solver_extra'])
    wait_ready('solver', SOLVER_PORT)
    start_vllm('vision', P['vision'], VISION_PORT, 0.40, P['vision_len'], vision=True)
    wait_ready('vision', VISION_PORT)
    sh(f'python -m mcqft.pipeline_ft generate --prep "{WORK}" --pdf "{PDF}" --n {N_QUESTIONS} '
       f'--bloom "{BLOOM}" --distractors {DISTRACTORS} --online-url http://127.0.0.1:{WRITER_PORT}/v1 '
       f'--online-model aqg-lora --tokenizer {BASE_MODEL} --out "{WORK}/questions_online"',
       log=f'{SAVE}/generate_online.log')
    shutil.copytree(f'{WORK}/questions_online', f'{SAVE}/questions_online', dirs_exist_ok=True)
    print(open(f'{WORK}/questions_online/summary.json', encoding='utf-8').read())

## Xử lý sự cố

| Triệu chứng | Cách xử lý |
|---|---|
| Máy chủ thoát, log có `CUDA out of memory` | `stop_all()`, đổi `PRESET` sang `fast`, chạy lại từ ô 2 |
| `CẢNH BÁO: kho câu nháp mỏng` ở ô 11 | tăng `OVERSAMPLE`, hoặc xem `draft_summary.json` để biết bị lọc vì lý do gì |
| Writer báo `hết câu nháp cho mức ...` trong log ô 14 | như trên: kho không đủ câu cho mức Bloom đó |
| Nhiều câu bị loại vì `question_too_trivial` | mô hình fine-tune soạn câu quá ngắn: giảm tỉ lệ mức "Nhận biết" trong `BLOOM`, hoặc huấn luyện thêm với dữ liệu khó |
| Nhiều câu bị loại vì `grounding` thấp | bản chép tài liệu kém (xem ô 9): tăng `--dpi` ở ô 8 hoặc dùng `PRESET='quality'` |
| Không câu nào có trích dẫn nguồn hợp lệ | tài liệu chỉ gồm đề bài (không có phần lí thuyết/công thức) nên mọi đoạn đều bị coi là "đề có sẵn" |
| Nhãn kiểm chứng toàn `non_verifiable` | đúng như thiết kế: Writer fine-tune không sinh biểu thức kiểm chứng, nhãn chỉ đến từ hội đồng giải độc lập |